# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [8]:
"""Title: Ranking Signal Decline: What Predicts It, and What Doesn't.
Abstract: This study asks whether pre decision signals which are search position, content depth, and content age can identify which pages in a content portfolio are losing search performance, without relying on the outcome data used to define decline itself. Using a March 2026 slice of FlyRank's content performance warehouse, a client grouped Random Forest classifier was compared against a rule based baseline and a logistic regression baseline on identical validation splits. The grouped model showed measured discriminative ability but relied heavily on one feature later flagged as a soft leakage risk, and its predicted probabilities rarely exceeded 0.53, indicating low absolute confidence even where relative ranking was useful. The resulting output is framed as a decision support ranking for human review, not an autonomous classifier, with explicit no go cases for where it should not be trusted.
Introduction / Problem Statement: Content teams cannot manually review every page in a growing portfolio to judge whether it's still performing. This study asks a page level question: using only signals known before a review decision is made, can a model usefully rank which individual pages are declining right now without quietly training on the outcome it's meant to predict? The decision this supports is editorial triage; the action is a targeted review, the cost of a wrong call is wasted reviewer hours or a real decline left unaddressed."""

'Title: Ranking Signal Decline: What Predicts It, and What Doesn\'t."\nAbstract: This study asks whether pre decision signals which are search position, content depth, and content age can identify which pages in a content portfolio are losing search performance, without relying on the outcome data used to define decline itself. Using a March 2026 slice of FlyRank\'s content performance warehouse, a client grouped Random Forest classifier was compared against a rule based baseline and a logistic regression baseline on identical validation splits. The grouped model showed measured discriminative ability but relied heavily on one feature later flagged as a soft leakage risk, and its predicted probabilities rarely exceeded 0.53, indicating low absolute confidence even where relative ranking was useful. The resulting output is framed as a decision support ranking for human review, not an autonomous classifier, with explicit no go cases for where it should not be trusted.\nIntroduction / Pro

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [9]:
"""Data: FlyRank ML Internship warehouse release, March 2026 development slice. 9,841,378 rows, 331,437 distinct content items, grain of one row per page per day. GA4 coverage in this slice: 4.2% of rows. Excluded: the sealed final month, GA4 fields where unavailable, and any column used to construct the label. No client names, URLs, or search queries appear anywhere in the study."""

'Data: FlyRank ML Internship warehouse release, March 2026 development slice. 9,841,378 rows, 331,437 distinct content items, grain of one row per page per day. GA4 coverage in this slice: 4.2% of rows. Excluded: the sealed final month, GA4 fields where unavailable, and any column used to construct the label. No client names, URLs, or search queries appear anywhere in the study.'

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [10]:
"""Methodology: Proxy label is_declining built by comparing first half vs. second half average daily clicks within the month an internal proxy, not ground truth. Five pre decision features used: average position, word count, content age, content type, main intent, plus impressions. Baseline: a rule flagging pages ranked 4–20 with below-average CTR for their tier. Validation: client-grouped split, to avoid the model learning client identity instead of a generalizable signal. Leakage checks: a deliberate trap confirmed AUC would jump toward near-perfect if leakage were present; that column was removed and never used in the reported model."""

'Methodology: Proxy label is_declining built by comparing first half vs. second half average daily clicks within the month an internal proxy, not ground truth. Five pre decision features used: average position, word count, content age, content type, main intent, plus impressions. Baseline: a rule flagging pages ranked 4–20 with below-average CTR for their tier. Validation: client-grouped split, to avoid the model learning client identity instead of a generalizable signal. Leakage checks: a deliberate trap confirmed AUC would jump toward near-perfect if leakage were present; that column was removed and never used in the reported model.'

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [11]:
"Results: Random Forest outperformed both baselines on every random seed tested (AUC 0.785–0.850) versus logistic regression (AUC 0.670) and the rule baseline. The seed to seed spread (0.065) was larger than the originally observed gap between a naive and grouped split (0.011), so that gap is reported as noise, not a stable finding. Feature importance leaned heavily on impressions, with position and word count contributing little consistent with the leakage caveat on impressions. Predicted probabilities were low confidence overall, which is why the final output is a relative ranking, not a hard classifier."

'Results: Random Forest outperformed both baselines on every random seed tested (AUC 0.785–0.850) versus logistic regression (AUC 0.670) and the rule baseline. The seed to seed spread (0.065) was larger than the originally observed gap between a naive and grouped split (0.011), so that gap is reported as noise, not a stable finding. Feature importance leaned heavily on impressions, with position and word count contributing little consistent with the leakage caveat on impressions. Predicted probabilities were low confidence overall, which is why the final output is a relative ranking, not a hard classifier.'

## 5. Limitations

*What this work cannot claim.*

In [12]:
"Limitations & Honest Framing: Observational only, no causal claims. Single month development window; the sealed June 2026 month has never been checked. Sparse GA4 coverage (4.2%) limited use of engagement features. A data quality issue was found where content_updated_date was dated after the report date for most rows, making staleness unusable in this slice specifically. All claims are phrased as observed, measured, or directional rather than proven."

'Limitations & Honest Framing: Observational only, no causal claims. Single month development window; the sealed June 2026 month has never been checked. Sparse GA4 coverage (4.2%) limited use of engagement features. A data quality issue was found where content_updated_date was dated after the report date for most rows, making staleness unusable in this slice specifically. All claims are phrased as observed, measured, or directional rather than proven.'

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [13]:
"""VISIBLE_BUT_AT_RISK is CTR/snippet review - protect existing page-one visibility first.
STRIKING_DISTANCE_DECLINING is content refresh - push positions 11–20 toward page one.
LOW_VOLUME_UNPROVEN  - we monitor only insufficient volume (52% of pages) to trust the signal; needs a separate lighter process.
STABLE_HEALTHY / WATCH_LIST - no action / quarterly review."""

'VISIBLE_BUT_AT_RISK is CTR/snippet review - protect existing page-one visibility first.\nSTRIKING_DISTANCE_DECLINING is content refresh - push positions 11–20 toward page one.\nLOW_VOLUME_UNPROVEN  - we monitor only insufficient volume (52% of pages) to trust the signal; needs a separate lighter process.\nSTABLE_HEALTHY / WATCH_LIST - no action / quarterly review.'

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [16]:
%pip -q install duckdb huggingface_hub matplotlib
import duckdb
import pandas as pd
from google.colab import userdata
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

DEV_MONTH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# --- labels ---
labels = con.sql(f"""
    SELECT content_hash_id,
      AVG(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_clicks END) AS clicks_first_half,
      AVG(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_clicks END) AS clicks_second_half
    FROM read_parquet('{DEV_MONTH_PATH}')
    GROUP BY content_hash_id
""").df()
labels['is_declining'] = (labels['clicks_second_half'] < labels['clicks_first_half']).astype(int)

# --- model_df ---
model_df = con.sql(f"""
    SELECT
      f.content_hash_id, f.client_hash_id,
      AVG(f.gsc_avg_position) AS avg_position,
      SUM(f.gsc_clicks) AS clicks_total,
      SUM(f.gsc_impressions) AS impressions_total,
      MAX(f.report_date) AS last_report_date
    FROM read_parquet('{DEV_MONTH_PATH}') f
    GROUP BY f.content_hash_id, f.client_hash_id
""").df()

dim = con.sql(f"""
    SELECT content_hash_id, content_type, word_count, main_intent, content_created_date
    FROM read_parquet('{DIM_CONTENT}')
""").df()

model_df = model_df.merge(dim, on='content_hash_id').merge(
    labels[['content_hash_id','is_declining']], on='content_hash_id'
)
model_df['content_age_days'] = (
    pd.to_datetime(model_df['last_report_date']) - pd.to_datetime(model_df['content_created_date'])
).dt.days
model_df = model_df.dropna(subset=['avg_position','word_count','content_age_days','client_hash_id'])

num_features = ['avg_position','word_count','content_age_days','impressions_total']
cat_features = ['content_type','main_intent']

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

preproc = ColumnTransformer([
    ('num', 'passthrough', num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

# --- grouped split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]
X_tr, y_tr = train_df[num_features + cat_features], train_df['is_declining']
X_te, y_te = test_df[num_features + cat_features], test_df['is_declining']

# --- logistic regression baseline (this is what defines logit_auc) ---
logit = Pipeline([('prep', preproc), ('clf', LogisticRegression(max_iter=1000))])
logit.fit(X_tr, y_tr)
logit_auc = roc_auc_score(y_te, logit.predict_proba(X_te)[:,1])

# --- random forest (grouped) ---
rf_grouped = Pipeline([('prep', preproc), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=0))])
rf_grouped.fit(X_tr, y_tr)
grouped_auc = roc_auc_score(y_te, rf_grouped.predict_proba(X_te)[:,1])
test_df = test_df.copy()
test_df['rf_prob'] = rf_grouped.predict_proba(X_te)[:,1]
test_df['rf_pred'] = rf_grouped.predict(X_te)

# --- archetype assignment (from your fixed, percentile-based version) ---
p90 = test_df['rf_prob'].quantile(0.90)
p75 = test_df['rf_prob'].quantile(0.75)
p50 = test_df['rf_prob'].quantile(0.50)

def assign_archetype(row):
    if row['impressions_total'] < 50:
        return 'LOW_VOLUME_UNPROVEN'
    if row['avg_position'] <= 10 and row['rf_prob'] >= p90:
        return 'VISIBLE_BUT_AT_RISK'
    elif 10 < row['avg_position'] <= 20 and row['rf_prob'] >= p75:
        return 'STRIKING_DISTANCE_DECLINING'
    elif row['rf_prob'] < p50:
        return 'STABLE_HEALTHY'
    else:
        return 'WATCH_LIST'

test_df['archetype'] = test_df.apply(assign_archetype, axis=1)

print(f"logit_auc = {logit_auc:.3f}")
print(f"grouped_auc (seed 0) = {grouped_auc:.3f}")
print(test_df['archetype'].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logit_auc = 0.657
grouped_auc (seed 0) = 0.847
archetype
LOW_VOLUME_UNPROVEN            12878
WATCH_LIST                      7108
VISIBLE_BUT_AT_RISK             1769
STABLE_HEALTHY                  1680
STRIKING_DISTANCE_DECLINING     1249
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.